# Refactoring PR Consistency Challenge — baseline

Kaggle: https://www.kaggle.com/competitions/refactoring-pr-consistency-challenge

Agent-authored Extract/Move refactoring PRs (AIDev, MSR 2026). Classify each PR from its title, body and code diff:
`Consistent`, `Inconsistent`, `Out of scope`, `Unavailable`. Metric: **macro F1 over the 4 classes**.
31 labelled train PRs (14 / 5 / 5 / 7), 14 test PRs. Data license CC BY 4.0.

This notebook: load → EDA → rule for `Unavailable` → repeated stratified CV → baselines
(majority, hand-made diff features, TF-IDF, frozen sentence embeddings, combinations) → submission.

In [1]:
import random, re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

SEED = 42
random.seed(SEED); np.random.seed(SEED)
LABELS = ["Consistent", "Inconsistent", "Out of scope", "Unavailable"]

DATA = Path("data")
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
for df in (train, test):
    df["body"] = df["body"].fillna("")
    df["code_changes"] = df["code_changes"].fillna("")
print(train.shape, test.shape, sample.shape)
train["label"].value_counts()

(31, 5) (14, 4) (14, 2)


label
Consistent      14
Unavailable      7
Out of scope     5
Inconsistent     5
Name: count, dtype: int64

## EDA

In [2]:
for df in (train, test):
    df["body_len"] = df["body"].str.len()
    df["code_len"] = df["code_changes"].str.len()
show = train[["id", "title", "body_len", "code_len", "label"]].sort_values("label")
print(show.to_string())
print("\nbody truncated at", train.body_len.max(), "chars; code at", train.code_len.max(), "chars")
print("test empty body & code:", ((test.body_len == 0) & (test.code_len == 0)).sum(), "of", len(test))

            id                                                                                         title  body_len  code_len         label
2   3255477552                                                 [BOT] refactor(Game): extract entity handlers       276      6000    Consistent
4   3211275635                                              refactor: move ApiResult to application services       357      6000    Consistent
6   3275715802                                                         Move EDIFACT processing into services       286      6000    Consistent
7   3165628544                                           [agent] feat(utils): extract token analysis helpers       291      4278    Consistent
12  2972920551   [refactor] Extract JournalEncounter-related functionality to UI/JournalEncounter.lua module       812      4745    Consistent
13  3215422687                                            Move FirebaseAuth logic to new AuthFirebaseService       383      5116    Consistent

**`Unavailable` = empty PR.** All 7 `Unavailable` train PRs have an empty body *and* an empty diff, and no other
class does (one `Inconsistent` PR has a body but no diff). So `Unavailable` is predicted by rule, and the models only
have to separate Consistent / Inconsistent / Out of scope on the remaining 24 PRs.

In [3]:
def is_unavailable(df):
    return (df["body"].str.strip() == "") & (df["code_changes"].str.strip() == "")

print(pd.crosstab(is_unavailable(train), train["label"]))

label  Consistent  Inconsistent  Out of scope  Unavailable
row_0                                                     
False          14             5             5            0
True            0             0             0            7


In [4]:
# One example per class
for lab in LABELS[:3]:
    r = train[train.label == lab].iloc[0]
    print(f"==================== {lab}: {r.title}")
    print(r.body[:600]); print("--- diff ---"); print(r.code_changes[:600], "\n")

==================== Consistent: [BOT] refactor(Game): extract entity handlers
## Summary
- refactor giant Game class by extracting movement and animation helpers
- document the new helper classes

## Testing
- `javac $(git ls-files '2006Scape Client/src/main/java/*.java')`


------
https://chatgpt.com/codex/tasks/task_e_68802b2f9014832b9fc18763963349f7
--- diff ---
Changed files (7): 2006Scape Client/src/main/java/core/EntityAnimationHandler.java, 2006Scape Client/src/main/java/core/EntityMovementHandler.java, 2006Scape Client/src/main/java/core/Game.java, docs/Client/classes/EntityAnimationHandler.md, docs/Client/classes/EntityMovementHandler.md, docs/Client/classes/Game.md, docs/Client/index.md
--- 2006Scape Client/src/main/java/core/EntityAnimationHandler.java (added, +130 -0)
@@ -0,0 +1,130 @@
+package core;
+
+import game.Entity;
+import game.NPC;
+import game.Player;
+import game.Animation;
+import game.SpotAnim;
+
+/**
+ * Handles entity facing an 

==================== Inconsi

## Hand-made diff features

The diff text starts with `Changed files (N): ...`, then one block per file: `--- path (status, +added -removed)`.
Features: file counts by status, line counts, share of docs/config files, title verbs, body length, and how much of the
description overlaps the diff (file names and `backticked` identifiers mentioned in the body that appear in the diff).

In [5]:
FILE_RE = re.compile(r"^--- (.+?) \((\w+), \+(\d+) -(\d+)\)", re.M)
DOC_EXT = (".md", ".txt", ".rst", ".json", ".yml", ".yaml", ".lock", ".toml", ".cfg", ".ini")

def diff_features(r):
    c, body, title = r.code_changes, r.body, r.title.lower()
    m = re.match(r"Changed files \((\d+)\): (.*)", c)
    files = [f.strip() for f in m.group(2).split(", ")] if m else []
    blocks = FILE_RE.findall(c)
    status = [b[1] for b in blocks]
    add = sum(int(b[2]) for b in blocks); rem = sum(int(b[3]) for b in blocks)
    text = (r.title + " " + body).lower()
    bases = [f.rsplit("/", 1)[-1].lower() for f in files]
    stems = [b.rsplit(".", 1)[0] for b in bases]
    idents = set(re.findall(r"`([A-Za-z_][\w.]{2,})`", body))
    return dict(
        n_files=len(files), log_files=np.log1p(len(files)),
        frac_added=np.mean([s == "added" for s in status]) if status else 0,
        frac_removed=np.mean([s == "removed" for s in status]) if status else 0,
        frac_renamed=np.mean([s == "renamed" for s in status]) if status else 0,
        frac_doc=np.mean([b.endswith(DOC_EXT) for b in bases]) if bases else 0,
        log_add=np.log1p(add), log_rem=np.log1p(rem),
        add_share=add / (add + rem) if add + rem else 0,
        diff_truncated=float(len(c) >= 6000), code_empty=float(c.strip() == ""),
        log_body=np.log1p(len(body)), body_empty=float(body.strip() == ""),
        n_bullets=body.count("\n- ") + body.count("\n* "),
        t_extract=float("extract" in title), t_move=float("move" in title),
        t_feat=float(title.startswith("feat") or "add" in title), t_refactor=float("refactor" in title),
        file_mention=np.mean([s in text for s in stems if len(s) > 2]) if stems else 0,
        ident_in_diff=np.mean([i in c for i in idents]) if idents else 0.5,
        n_idents=np.log1p(len(idents)),
    )

F_train = pd.DataFrame([diff_features(r) for r in train.itertuples()])
F_test = pd.DataFrame([diff_features(r) for r in test.itertuples()])
F_train.assign(label=train.label)[~is_unavailable(train).values].groupby("label").mean().T.round(2)

label,Consistent,Inconsistent,Out of scope
n_files,6.07,14.20,14.20
log_files,1.77,2.04,2.55
frac_added,0.49,0.00,0.23
frac_removed,0.03,0.00,0.00
frac_renamed,0.01,0.07,0.38
frac_doc,0.07,0.20,0.15
log_add,4.69,2.73,3.94
log_rem,4.25,2.83,3.31
add_share,0.61,0.39,0.63
diff_truncated,0.36,0.60,0.60


## Cross-validation setup

31 rows (only 5 per minority class), so a single split is meaningless: 5-fold stratified CV repeated 10 times,
macro F1 over all 4 classes, mean ± std across folds. In each fold the `Unavailable` rule is applied and the model is
fit only on the non-empty training PRs.

In [6]:
y = train["label"].values
unav_tr, unav_te = is_unavailable(train).values, is_unavailable(test).values
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEED)
SPLITS = list(cv.split(np.zeros(len(y)), y))
results, oof_first = {}, {}

def evaluate(name, fit_predict):
    """fit_predict(tr_idx, va_idx) -> labels for va_idx (non-empty rows only; rule handles Unavailable)."""
    scores, oof = [], np.empty(len(y), dtype=object)
    for k, (tr, va) in enumerate(SPLITS):
        pred = np.array(["Unavailable"] * len(va), dtype=object)
        tr_m, va_m = tr[~unav_tr[tr]], va[~unav_tr[va]]
        if len(va_m):
            pred[~unav_tr[va]] = fit_predict(tr_m, va_m)
        scores.append(f1_score(y[va], pred, labels=LABELS, average="macro", zero_division=0))
        if k < 5: oof[va] = pred
    results[name] = dict(macro_f1=np.mean(scores), std=np.std(scores))
    oof_first[name] = oof
    print(f"{name:40s} macro F1 {np.mean(scores):.3f} ± {np.std(scores):.3f}")

def lr_on(X, C=1.0):
    def fp(tr, va):
        m = make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight="balanced", max_iter=5000))
        return m.fit(X[tr], y[tr]).predict(X[va])
    return fp

## Baselines

In [7]:
evaluate("all Consistent (host benchmark)", lambda tr, va: np.array(["Consistent"] * len(va)))
evaluate("rule + Consistent", lambda tr, va: np.array(["Consistent"] * len(va)))

Xf = F_train.values
for C in (0.03, 0.1, 0.3, 1):
    evaluate(f"rule + diff features LR C={C}", lr_on(Xf, C))

all Consistent (host benchmark)          macro F1 0.433 ± 0.008
rule + Consistent                        macro F1 0.433 ± 0.008


rule + diff features LR C=0.03           macro F1 0.494 ± 0.113


rule + diff features LR C=0.1            macro F1 0.537 ± 0.143


rule + diff features LR C=0.3            macro F1 0.538 ± 0.152


rule + diff features LR C=1              macro F1 0.524 ± 0.139


In [8]:
def text_of(df, with_code=True):
    t = df["title"] + "\n" + df["body"]
    return t + "\n" + df["code_changes"] if with_code else t

def tfidf_lr(with_code, C):
    texts = text_of(train, with_code).values
    def fp(tr, va):
        m = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=20000),
                          LogisticRegression(C=C, class_weight="balanced", max_iter=5000))
        return m.fit(texts[tr], y[tr]).predict(texts[va])
    return fp

for wc in (False, True):
    for C in (1, 10):
        evaluate(f"rule + tfidf {'title+body+diff' if wc else 'title+body'} LR C={C}", tfidf_lr(wc, C))

rule + tfidf title+body LR C=1           macro F1 0.559 ± 0.116


rule + tfidf title+body LR C=10          macro F1 0.556 ± 0.125


rule + tfidf title+body+diff LR C=1      macro F1 0.460 ± 0.092


rule + tfidf title+body+diff LR C=10     macro F1 0.446 ± 0.076


## Frozen sentence embeddings

`all-mpnet-base-v2` mean-pooled embeddings of (a) title + body and (b) the diff, computed once for train+test
(no labels involved). Long diffs are split into 512-token chunks and averaged. Also the cosine between the description
and diff embeddings as a "does the text match the code" feature.

In [9]:
import torch
from transformers import AutoTokenizer, AutoModel
torch.manual_seed(SEED)
DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
MODEL = "sentence-transformers/all-mpnet-base-v2"
tok = AutoTokenizer.from_pretrained(MODEL)
enc = AutoModel.from_pretrained(MODEL).to(DEVICE).eval()

@torch.no_grad()
def embed(text, max_length=512):
    ids = tok(text, add_special_tokens=False)["input_ids"] or [tok.unk_token_id]
    chunks = [ids[i:i + max_length - 2] for i in range(0, len(ids), max_length - 2)]
    vecs = []
    for ch in chunks:
        ids_t = torch.tensor([[tok.cls_token_id] + ch + [tok.sep_token_id]], device=DEVICE)
        b = {"input_ids": ids_t, "attention_mask": torch.ones_like(ids_t)}
        h = enc(**b).last_hidden_state.mean(1)
        vecs.append(torch.nn.functional.normalize(h, dim=-1))
    return torch.cat(vecs).mean(0).cpu().numpy()

def embed_df(df):
    d = np.stack([embed(t) for t in text_of(df, with_code=False)])
    c = np.stack([embed(t if t else "(empty)") for t in df["code_changes"]])
    return d, c

D_train, C_train = embed_df(train)
D_test, C_test = embed_df(test)
cos = lambda a, b: (a * b).sum(1) / np.linalg.norm(a, axis=1) / np.linalg.norm(b, axis=1)
F_train["desc_code_cos"] = cos(D_train, C_train); F_test["desc_code_cos"] = cos(D_test, C_test)
print(D_train.shape, C_train.shape)
F_train.assign(label=train.label)[~unav_tr].groupby("label")["desc_code_cos"].describe().round(3)

/Users/home/workspace/School/ML/competition/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1121 16:49:06.819000 66160 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6003.15it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (523 > 512). Running this sequence through the model will result in indexing errors


(31, 768) (31, 768)


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
Consistent,14.0,0.615,0.090,0.431,0.592,0.623,0.647,0.775
Inconsistent,5.0,0.429,0.260,0.059,0.381,0.443,0.475,0.789
Out of scope,5.0,0.554,0.106,0.452,0.486,0.502,0.636,0.696


In [10]:
Xf = F_train.values   # now includes desc_code_cos
for C in (0.03, 0.1):
    evaluate(f"rule + diff features+cos LR C={C}", lr_on(Xf, C))
for name, X in [("desc emb", D_train), ("desc+code emb", np.hstack([D_train, C_train])),
                ("desc+code emb + features", np.hstack([D_train, C_train, Xf]))]:
    for C in (0.01, 0.1):
        evaluate(f"rule + {name} LR C={C}", lr_on(X, C))

rule + diff features+cos LR C=0.03       macro F1 0.517 ± 0.116


rule + diff features+cos LR C=0.1        macro F1 0.539 ± 0.140


rule + desc emb LR C=0.01                macro F1 0.607 ± 0.147


rule + desc emb LR C=0.1                 macro F1 0.609 ± 0.150


rule + desc+code emb LR C=0.01           macro F1 0.605 ± 0.175


rule + desc+code emb LR C=0.1            macro F1 0.606 ± 0.174


rule + desc+code emb + features LR C=0.01 macro F1 0.594 ± 0.156


rule + desc+code emb + features LR C=0.1 macro F1 0.594 ± 0.156


## Results

In [11]:
res = pd.DataFrame(results).T.sort_values("macro_f1", ascending=False)
res.round(3)

,macro_f1,std
rule + desc emb LR C=0.1,0.609,0.150
rule + desc emb LR C=0.01,0.607,0.147
rule + desc+code emb LR C=0.1,0.606,0.174
rule + desc+code emb LR C=0.01,0.605,0.175
rule + desc+code emb + features LR C=0.1,0.594,0.156
rule + desc+code emb + features LR C=0.01,0.594,0.156
rule + tfidf title+body LR C=1,0.559,0.116
rule + tfidf title+body LR C=10,0.556,0.125
rule + diff features+cos LR C=0.1,0.539,0.140
rule + diff features LR C=0.3,0.538,0.152


In [12]:
best = res.index[0]
print("best:", best)
oof = oof_first[best]; mask = np.array([o is not None for o in oof])
print(pd.DataFrame(confusion_matrix(y[mask], oof[mask].astype(str), labels=LABELS), index=LABELS, columns=LABELS))

best: rule + desc emb LR C=0.1
              Consistent  Inconsistent  Out of scope  Unavailable
Consistent            13             1             0            0
Inconsistent           3             2             0            0
Out of scope           1             2             2            0
Unavailable            0             0             0            7


## Submission

Fit the CV-best model on all non-empty training PRs, apply the `Unavailable` rule, predict the 14 test PRs.
Pick by CV, not by the public leaderboard (assignment rule).

In [13]:
BUILDERS = {
    "features": (F_train.values, F_test.values),
    "desc emb": (D_train, D_test),
    "desc+code emb": (np.hstack([D_train, C_train]), np.hstack([D_test, C_test])),
    "desc+code emb + features": (np.hstack([D_train, C_train, F_train.values]),
                                 np.hstack([D_test, C_test, F_test.values])),
}
def final_predict(name):
    if "tfidf" in name:
        wc = "diff" in name; C = float(name.split("C=")[1])
        m = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=20000),
                          LogisticRegression(C=C, class_weight="balanced", max_iter=5000))
        Xtr, Xte = text_of(train, wc).values, text_of(test, wc).values
    elif "Consistent" in name:
        return np.where(unav_te, "Unavailable", "Consistent")
    else:
        key = "features" if "diff features" in name else name.split("rule + ")[1].split(" LR")[0]
        Xtr, Xte = BUILDERS[key]; C = float(name.split("C=")[1])
        m = make_pipeline(StandardScaler(), LogisticRegression(C=C, class_weight="balanced", max_iter=5000))
    m.fit(Xtr[~unav_tr], y[~unav_tr])
    return np.where(unav_te, "Unavailable", m.predict(Xte))

pred = final_predict(best)
sub = pd.DataFrame({"id": test["id"], "label": pred})
assert list(sub.columns) == list(sample.columns) and set(sub.id) == set(sample.id)
assert set(sub.label) <= set(LABELS)
print(sub.assign(title=test.title).to_string())
sub.to_csv("submission_baseline.csv", index=False)
print("wrote submission_baseline.csv | model:", best, "| CV:", results[best])

            id         label                                                                           title
0   3071707121  Out of scope                                          Refactor: move futarchy core contracts
1   3165805307    Consistent                                    [Agent] Extract action definition processing
2   3124333790  Inconsistent                                             Move setting explanation to tooltip
3   3243478130  Inconsistent  [worker, trainer] refactor: Move config classes from trainer to workers module
4   3155881276    Consistent                         refactor(ethexe): extract fast sync as optional service
5   2761686760    Consistent                           refactor: move SumOrderComparator into separate files
6   2761418924    Consistent                           refactor: move SumOrderComparator into separate files
7   3119376417    Consistent                                                   refactor: extract app factory
8   3260210061    C